# G-reasoner on the team graph (`fin10k_v0`)

Runs the pretrained G-reasoner-34M graph retriever on the team's knowledge graph and scores it on the golden questions. Setup and details: `G-reasoner/README.md`.

**How to use:** put the repo in your Google Drive (README, "Setup"), open this notebook from Drive, Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. About 15 minutes the first time (install ~5 min, graph embedding ~8 min), about 8 minutes after that.

Where things live:
- **Code and team data**: your copy of the repo in Drive, used read-only here. Edit the code there (or on your computer if it syncs with Drive for desktop) and re-run the cell.
- **gfmrag working data, cache and results**: `My Drive/fin_greasoner_cache/`, so a new runtime skips the slow embedding step.

Success = section 8 prints `=== G-reasoner loaded ...` and ranked documents for each question; section 9 prints Hit@5 / Hit@10 and the folder the results were saved to.


## 1. Install Python 3.12
gfmrag needs Python 3.12, so it lives in its own venv at `/content/py312`, separate from Colab's own packages.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq python3.12 python3.12-venv
!python3.12 --version

## 2. Create the venv and install gfmrag + ninja
Slow (several minutes). gfmrag is pinned to 2.0.0, the version the scripts were written against.

In [ ]:
!python3.12 -m venv /content/py312
!/content/py312/bin/python -m pip install -q --upgrade pip
!/content/py312/bin/python -m pip install -q gfmrag==2.0.0 ninja
!/content/py312/bin/python -c "import gfmrag, ninja; print('gfmrag + ninja imported successfully')"

## 3. Put the venv on PATH
gfmrag steps run with the venv's Python (`PYTHON`). PATH must include the venv so ninja is found when G-reasoner compiles its CUDA kernel on first use.

In [ ]:
import os, subprocess

PYTHON = "/content/py312/bin/python"
os.environ["PATH"] = "/content/py312/bin:" + os.environ["PATH"]
print("ninja:", subprocess.run(["which", "ninja"], capture_output=True, text=True).stdout)

## 4. Patch gfmrag's PyG version parsing
gfmrag crashes on PyG versions like `2.8.0.post1`. This keeps only the first three parts.

In [ ]:
from pathlib import Path

path = Path("/content/py312/lib/python3.12/site-packages/gfmrag/models/ultra/layers.py")
text = path.read_text()
old = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")]'
new = 'pyg_version = [int(i) for i in torch_geometric.__version__.split(".")[:3]]'
if old in text:
    path.write_text(text.replace(old, new))
    print("Patch applied.")
elif new in text:
    print("Patch already applied.")
else:
    print("Original line not found. Check layers.py manually.")

## 5. Mount Google Drive and find the repo
Colab asks for permission the first time. The cell looks for the repo in the usual places (uploaded to My Drive, synced from a computer with Google Drive for desktop, or cloned into this runtime); if it finds none or several, set `REPO` at the top of the cell. gfmrag's working data and results go to `My Drive/fin_greasoner_cache/` either way.

In [ ]:
import glob
from google.colab import drive
drive.mount("/content/drive")

# Where is the repo? Leave REPO = None to search the usual places, or set it yourself, e.g.
#   REPO = "/content/drive/MyDrive/financial-graphrag"   (repo folder uploaded to My Drive)
#   REPO = "/content/financial-graphrag"                 (cloned into this runtime with git)
REPO = None
if REPO is None:
    patterns = ["/content/drive/MyDrive/financial-graphrag",            # uploaded to My Drive
                "/content/drive/MyDrive/*/financial-graphrag",
                "/content/drive/MyDrive/*/*/financial-graphrag",
                "/content/drive/Othercomputers/*/*/financial-graphrag",  # Drive for desktop sync
                "/content/drive/Othercomputers/*/financial-graphrag",
                "/content/financial-graphrag"]                          # git clone in Colab
    found = sorted({p for pat in patterns for p in glob.glob(pat) if os.path.isdir(f"{p}/G-reasoner")})
    print("repo copies found:", found or "NONE")
    assert len(found) == 1, "Set REPO above to your copy of the repo (see G-reasoner/README.md, 'Setup')."
    REPO = found[0]
CODE = f"{REPO}/G-reasoner"
print("using repo:", REPO)

CACHE = "/content/drive/MyDrive/fin_greasoner_cache"  # gfmrag data root + results (survives runtime resets)
DATA  = f"{CACHE}/data"
os.makedirs(DATA, exist_ok=True)

need = [f"{CODE}/{f}" for f in ["prepare_data.py", "entity_index.py", "gfm_components.py", "index_graph.py", "query.py"]]
need += [f"{REPO}/data/graph/fin10k_v0/processed/stage1/{f}" for f in ["nodes.csv", "relations.csv", "edges.csv"]]
missing = [p.replace(REPO + "/", "") for p in need if not os.path.exists(p)]
print("missing files:", missing or "none (if some are missing, Drive may still be uploading)")

## 6. Prepare the graph
Copies `data/graph/fin10k_v0/processed/stage1` from the clone into `DATA` and writes `raw/documents.json`, which gfmrag requires even when stage1 exists. If the team graph changed since the last run, the old embedding cache is removed.

In [ ]:
!python "{CODE}/prepare_data.py" --repo "{REPO}" --graph fin10k_v0 --root "{DATA}"

In [ ]:
!nvidia-smi

## 7. Embed the graph (stage2)
First run: downloads Qwen3-Embedding-0.6B and embeds all 3,618 nodes (a few minutes). Later runs find the cache on Drive and finish quickly. This runs on its own so the query step starts only one vLLM model on the T4.

In [ ]:
!{PYTHON} "{CODE}/index_graph.py" --root "{DATA}" --data-name fin10k_v0

## 8. Load G-reasoner and retrieve
Start nodes come from `entity_index.py` (alias matching on graph node names, no LLM). The first run also compiles G-reasoner's CUDA kernel, which takes a minute or two.

In [ ]:
!{PYTHON} "{CODE}/query.py" --root "{DATA}" --data-name fin10k_v0 --top-k 5 \
    -q "What were Apple's total net sales in fiscal 2025?" \
    -q "How much DARZALEX revenue did J&J report in FY2025?" \
    -q "Which spent more on research and development in fiscal 2025, Apple or Microsoft?" \
    -q "Tell me about cybersecurity" \
    --out "{CACHE}/probe_results.jsonl"

## 9. Score on the golden set
Runs the team's golden questions whose gold chunks are in the graph (11 of 19 for `fin10k_v0`), scored with the vector baseline's own metrics (`vector_rag_baseline/rag/metrics.py`). Results go to `My Drive/fin_greasoner_cache/results/`, in the baseline's `per_question.jsonl` / `summary.json` format.

In [ ]:
import time
RUN = f"{CACHE}/results/{time.strftime('%Y%m%d-%H%M%S')}"
!{PYTHON} "{CODE}/evaluate.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v0 --out "{RUN}" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING|NCCL"

## 10. Diagnose retrieval
Runs the covered golden questions four ways in one model load: all start nodes, without the Company/FiscalYear hub nodes, hubs only, and plain embedding similarity with no graph (`dense_qwen3`).

In [ ]:
import time
RUN = f"{CACHE}/results/diagnose-{time.strftime('%Y%m%d-%H%M%S')}"
!{PYTHON} "{CODE}/diagnose.py" --repo "{REPO}" --root "{DATA}" --data-name fin10k_v0 --out "{RUN}" 2>&1 | grep -v -E "EngineCore|^INFO|Gloo|WARNING|NCCL|torch_dtype|cpp_extension"